# Classification binaire PFAS — Somme vs somme des seuils individuels

**Objectif :** Prédire si la **charge PFAS totale** d'un puits californien dépasse la
**somme des seuils individuels** définis dans le protocole multilabel (notebook 03).

## Définition de la cible

Au lieu de tester chaque composé contre son propre seuil (EPA 2024), on agrège :

$$ \text{cible} = 1 \quad \text{si} \quad \sum_{i=1}^{27} C_i \;\geq\; \sum_{i=1}^{27} s_i $$

- $C_i$ : concentration mesurée du composé $i$ (27 PFAS cibles du notebook 03)
- $s_i$ : seuil individuel du composé $i$ (EPA 2024 NPDWR si défini, sinon 2 ng/L)
- $\sum s_i = 74\ \text{ng/L}$ — **somme des 27 seuils**

| Famille de seuil | Composés | Seuil (ng/L) |
|------------------|----------|--------------|
| MCL EPA 2024 | PFOA, PFOS | 4 |
| MCL EPA 2024 | PFHxS, PFNA | 10 |
| Défaut (MDL) | 23 autres PFAS | 2 |
| **Somme** | **27 composés** | **74** |

> Note : `sum_seuil = 74 ng/L` est proche de l'ancien seuil EPA Health Advisory (70 ng/L),
> mais s'en distingue : il agrège **27 composés** (pas seulement PFOA+PFOS) avec des seuils
> hétérogènes issus du protocole multilabel.

## Protocole (identique au notebook 01)
- Mêmes features, même retrait de la localisation pure (`DROP_LOCATION`, Dong et al.)
- Mêmes modèles : **Random Forest** + **XGBoost**, optimisés par **Optuna**
- Seule la **construction de la cible** change.

## Dataset
- Source : `CA-PFAS-ASGWS.parquet` — 46 338 échantillons
- Balance : ~22.3% positifs (1:3.5) → plus déséquilibrée que l'EPA 2024 (1:1.2),
  d'où l'importance de `class_weight="balanced"` (RF) et `scale_pos_weight` (XGBoost).

## 0. Imports et configuration

In [1]:
import warnings
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve, accuracy_score,
    recall_score, precision_score, balanced_accuracy_score,
)

import xgboost as xgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ── Chemins ──────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"
FIGURES_DIR   = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
RANDOM_STATE = 42
print("Imports OK")

Imports OK


## 1. Chargement des données

In [2]:
df = pd.read_parquet(PROCESSED_DIR / "CA-PFAS-ASGWS.parquet")
print(f"Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(3)

Dataset : 46,338 lignes × 201 colonnes


,gm_well_id,collection_date,ADONA_ngL,F53B_major_ngL,F53B_minor_ngL,FTS_4_2_ngL,FTS_6_2_ngL,FTS_8_2_ngL,HFPO_DA_ngL,NEtFOSAA_ngL,...,runoff_mm,soil_moi_0_10_kg_m2,soil_moi_10_40_kg_m2,soil_moi_40_100_kg_m2,soil_moi_100_200_kg_m2,root_zone_moist_kg_m2,temp_c,snowpack_mm,gldas_dist_km,soil_moisture_total_mm
0,100001,2019-03-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.35,...,0.000506,23.504955,73.092522,149.431152,239.711273,50.487106,15.603387,0.0,14.443,485.739902
1,100001,2020-04-28,1.0,1.0,1.0,NaN,NaN,NaN,2.5,1.00,...,0.000722,23.421059,72.313629,145.745392,232.140076,45.648960,17.293207,0.0,14.443,473.620155
2,100002,2019-03-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.35,...,0.000506,23.504955,73.092522,149.431152,239.711273,50.487106,15.603387,0.0,14.443,485.739902


## 2. Construction de la cible EPA 2024

In [3]:
# ── Seuils individuels — importés du protocole multilabel (notebook 03) ───────
import sys
sys.path.insert(0, "..")
from src.ml_multilabel import DETECTION_THRESHOLDS, DEFAULT_THRESHOLD, PFAS_TARGET_COLS

# Somme des seuils individuels (27 composés)
SUM_SEUIL = sum(DETECTION_THRESHOLDS.get(c, DEFAULT_THRESHOLD) for c in PFAS_TARGET_COLS)

def compute_target_sum_threshold(df):
    """cible = 1 si la somme des 27 PFAS cibles >= somme de leurs seuils."""
    present = [c for c in PFAS_TARGET_COLS if c in df.columns]
    sum_conc = df[present].fillna(0).sum(axis=1)
    return (sum_conc >= SUM_SEUIL).astype(int), sum_conc

df["target"], _sum_conc = compute_target_sum_threshold(df)

# ── Détail des seuils ────────────────────────────────────────────────────────
n_present = sum(1 for c in PFAS_TARGET_COLS if c in df.columns)
n_mcl = sum(1 for c in PFAS_TARGET_COLS if c in DETECTION_THRESHOLDS
            and DETECTION_THRESHOLDS[c] != DEFAULT_THRESHOLD)
print(f"Composés agrégés      : {n_present}/27")
print(f"  dont seuils MCL EPA  : {n_mcl}  (PFOA/PFOS=4, PFHxS/PFNA=10 ng/L)")
print(f"  dont seuils défaut   : {n_present - n_mcl}  ({DEFAULT_THRESHOLD} ng/L)")
print(f"SOMME DES SEUILS       : sum_seuil = {SUM_SEUIL:.0f} ng/L")
print()

n_pos = df["target"].sum()
n_neg = len(df) - n_pos
ratio = n_neg / n_pos
print(f"Positifs (Σ ≥ {SUM_SEUIL:.0f}) : {n_pos:,}  ({100*n_pos/len(df):.1f}%)")
print(f"Négatifs (Σ < {SUM_SEUIL:.0f}) : {n_neg:,}  ({100*n_neg/len(df):.1f}%)")
print(f"Ratio négatif/positif : 1:{ratio:.2f}")

Composés agrégés      : 27/27
  dont seuils MCL EPA  : 4  (PFOA/PFOS=4, PFHxS/PFNA=10 ng/L)
  dont seuils défaut   : 23  (2.0 ng/L)
SOMME DES SEUILS       : sum_seuil = 74 ng/L

Positifs (Σ ≥ 74) : 10,334  (22.3%)
Négatifs (Σ < 74) : 36,004  (77.7%)
Ratio négatif/positif : 1:3.48


In [4]:
# Distribution de la charge PFAS totale + comparaison avec d'autres cibles
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Graphe 1 : histogramme de la somme des 27 PFAS (échelle log) + seuil
sum_log = np.log10(_sum_conc + 1)
axes[0].hist(sum_log, bins=70, color="#3498db", alpha=0.85)
axes[0].axvline(np.log10(SUM_SEUIL + 1), color="#c0392b", ls="--", lw=2,
                label=f"sum_seuil = {SUM_SEUIL:.0f} ng/L")
axes[0].set_xlabel("log₁₀(Σ 27 PFAS + 1)  [ng/L]")
axes[0].set_ylabel("Nombre d'échantillons")
axes[0].set_title("Charge PFAS totale — seuil de décision")
axes[0].legend()

# Graphe 2 : comparaison de 3 définitions de cible
mcls = {"PFOA_ngL":4,"PFOS_ngL":4,"PFNA_ngL":10,"PFHxS_ngL":10,"HFPO_DA_ngL":10}
hi_refs = {"PFNA_ngL":10,"PFHxS_ngL":10,"HFPO_DA_ngL":10,"PFBS_ngL":2000}
epa = pd.Series(False, index=df.index)
for c,m in mcls.items():
    if c in df.columns: epa |= df[c].fillna(0) > m
epa |= sum(df[c].fillna(0)/r for c,r in hi_refs.items() if c in df.columns) > 1
old70 = (df["sum_pfas_ngL"].fillna(0) > 70)

comparison = pd.DataFrame({
    "Cible": [f"Σ27 ≥ {SUM_SEUIL:.0f} (ce NB)", "EPA 2024\n(MCL+HI)", "Ancienne\nΣ>70"],
    "Positifs": [int(df["target"].sum()), int(epa.sum()), int(old70.sum())],
    "Taux": [100*df["target"].mean(), 100*epa.mean(), 100*old70.mean()],
})
bars = axes[1].bar(comparison["Cible"], comparison["Positifs"],
                   color=["#e74c3c", "#95a5a6", "#bdc3c7"], edgecolor="white", width=0.6)
for bar, taux in zip(bars, comparison["Taux"]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f"{taux:.1f}%", ha="center", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Nombre de positifs")
axes[1].set_title("Comparaison des définitions de cible")
axes[1].set_ylim(0, 26000)

plt.suptitle("Cible — Somme PFAS vs somme des seuils individuels", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "sumthr_target_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Ingénierie des features

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG — protocole anti-fuite spatiale (Dong et al. 2024)
# ══════════════════════════════════════════════════════════════════════════════
#  DROP_LOCATION : retire les attributs de LOCALISATION PURE (lat/lon, county,
#  bassins…) qui permettent au modèle de MÉMORISER la géographie plutôt que
#  d\'apprendre les mécanismes environnementaux.
#
#  Analyse de robustesse (RF 300 arbres, dataset complet) :
#     Scénario                          #feat   AUC      F1
#     A. Tout (localisation incluse)      92   0.9745  0.918
#     B. Sans localisation pure  <- ICI   86   0.9656  0.899   <- recommandé mémoire
#     C. Sans loc + sans contexte échant. 84   0.9633  0.895
#
#  L\'inflation par mémorisation spatiale est réelle mais modeste (~1 pt AUC) :
#  l\'essentiel du signal vient de la PROXIMITÉ AUX SOURCES (geotracker), qui est
#  mécanistique et donc CONSERVÉE même en scénario B.
DROP_LOCATION = True   # <- False pour reproduire le scénario A (notebook d\'origine)

# ── Features temporelles ─────────────────────────────────────────────────────
dt = pd.to_datetime(df["collection_date"])
df["year"]   = dt.dt.year
df["month"]  = dt.dt.month
df["season"] = ((dt.dt.month % 12) // 3 + 1).astype(int)  # 1=hiver 2=printemps 3=été 4=automne

# ── Colonnes à exclure — 3 types de fuite + bruit ────────────────────────────
#
#  1. Concentrations PFAS (_ngL)
#     -> Encodent directement la valeur mesurée, source de la cible
#
#  2. Indicateurs de détection binaires (_detected, label_)
#     -> "PFOA_detected=1" signifie PFOA détecté au-dessus du seuil analytique
#        corrélation avec target : PFOA_detected=0.789, PFOS_detected=0.779
#        Quand _detected=True : 89% des échantillons dépassent le MCL
#        -> Fuite directe, même nature que les concentrations
#
#  3. Ancienne cible (target_sum_gt70)
#     -> Corrélée à 0.59 avec EPA 2024 target
#
#  4. Bruit spatial : gldas_dist_km
#     -> Distance au pixel GLDAS le plus proche (max=17 km), artefact de grille
#        corr=+0.027 avec target -> aucune valeur prédictive

LEAK_SUFFIXES  = ("_ngL",)
LEAK_BOOL_SUFFIXES = ("_detected",)   # indicateurs de détection PFAS
LEAK_PREFIXES  = ("label_",)
LEAK_EXACT     = {
    "sum_pfas_ngL", "pfas_class_assignment", "gm_well_id",
    "collection_date", "target", "target_sum_gt70",
    "gldas_dist_km",  # artefact de grille GLDAS, corr=0.027
}

# Colonnes >90% null — imputation médiane ne suffira pas, trop de signal artificiel
HIGH_NULL_DROP = {
    "cocontam_xylenes",            # 100% null
    "cocontam_tmb124",             # 99% null
    "cocontam_dce12c",             # 99% null
    "cocontam_btbzt",              # 99% null
    "soil_gradation_uniformity",   # 98% null (CA manque siltco_r)
    "soil_gradation_curvature",    # 98% null
    "soil_silt_coarse_pct",        # 95% null
    "soil_silt_fine_pct",          # 95% null
    "cocontam_no3n",               # 95% null
    "well_depth_ft",               # 95% null (non renseigné GAMA)
}

# Identifiants géographiques redondants (déjà encodés par county/regional_board)
DROP_REDUNDANT = {"dwr_basin", "sgma_basin_name", "sgma_subbasin_name"}

# ── Localisation PURE (protocole Dong et al.) — retirée si DROP_LOCATION ──────
#  Conserve volontairement les features de PROXIMITÉ aux sources (geotracker) :
#  elles sont mécanistiques (transport depuis une source), pas positionnelles.
LOCATION_FEATURES = {
    "latitude", "longitude",
    "county", "regional_board", "dwr_region", "sgma_region_office",
}
loc_drop = LOCATION_FEATURES if DROP_LOCATION else set()

# ── Catégorielles sélectionnées ──────────────────────────────────────────────
CATEGORICAL_FEATURES = [
    "gm_well_category",        # Type de puits : Municipal/Monitoring/Domestic…
    "nearest_geotracker_type", # Type du site PFAS le plus proche (industrie, armée, feu…)
    "soil_texture_class",      # Classe texturale USDA (proxy perméabilité)
    "sgma_region_office",      # Bureau régional SGMA (basin adjudication)
    "gm_dataset_name",         # Programme surveillance GAMA
    "county",                  # Comté (58 uniques — proxy land-use/industrial)
    "regional_board",          # Tableau régional eau (9 boards)
    "dwr_region",              # Région DWR (10 régions)
]
cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns and c not in loc_drop]

# ── Numériques ───────────────────────────────────────────────────────────────
num_cols = [
    c for c in df.columns
    if df[c].dtype.kind in ("f", "i", "u")
    and not any(c.endswith(s) for s in LEAK_SUFFIXES)
    and not any(c.endswith(s) for s in LEAK_BOOL_SUFFIXES)
    and not any(c.startswith(p) for p in LEAK_PREFIXES)
    and c not in LEAK_EXACT
    and c not in HIGH_NULL_DROP
    and c not in DROP_REDUNDANT
    and c not in loc_drop
    and c not in cat_cols
]

feature_names = num_cols + cat_cols

# ── Résumé des exclusions ────────────────────────────────────────────────────
n_detected  = sum(1 for c in df.columns if c.endswith("_detected"))
n_label     = sum(1 for c in df.columns if c.startswith("label_"))
n_ngL       = sum(1 for c in df.columns if c.endswith("_ngL"))

print("════════════════════════════════════════════════════════")
print(f"  Exclusions (fuite + bruit)   |  DROP_LOCATION = {DROP_LOCATION}")
print("════════════════════════════════════════════════════════")
print(f"  _ngL   (concentrations)      : {n_ngL} colonnes")
print(f"  label_ (détection PFAS)      : {n_label} colonnes")
print(f"  _detected (détection PFAS)   : {n_detected} colonnes  <- corr jusqu\'à 0.79")
print(f"  target_sum_gt70              : 1 colonne   <- ancienne cible (corr=0.59)")
print(f"  gldas_dist_km                : 1 colonne   <- bruit artefact grille")
print(f"  >90% null                    : {len(HIGH_NULL_DROP)} colonnes")
print(f"  redondantes géo              : {len(DROP_REDUNDANT)} colonnes")
if DROP_LOCATION:
    print(f"  localisation pure (Dong)     : {len(LOCATION_FEATURES)} colonnes  <- {', '.join(sorted(LOCATION_FEATURES))}")
print("────────────────────────────────────────────────────────")
print(f"  Features numériques retenues : {len(num_cols)}")
print(f"  Features catégorielles       : {len(cat_cols)}")
print(f"  TOTAL                        : {len(feature_names)}")
print("════════════════════════════════════════════════════════")
print()
print("Groupes de features retenues :")
groups = {
    "SPATIAL":        [c for c in num_cols if c in ["latitude","longitude"]],
    "GEOTRACKER":     [c for c in num_cols if "geotracker" in c],
    "GLDAS_HYDRO":    [c for c in num_cols if any(x in c for x in ["rainfall","et_mm","runoff","soil_moi","root_zone","temp_c","snowpack","soil_moisture_total"])],
    "SSURGO_SOL":     [c for c in num_cols if c.startswith("soil_") and c not in ["soil_moi_0_10_kg_m2","soil_moi_10_40_kg_m2","soil_moi_40_100_kg_m2","soil_moi_100_200_kg_m2","soil_moisture_total_mm"]],
    "AQS_AIR":        [c for c in num_cols if c.startswith("aqs_")],
    "CO_CONTAM":      [c for c in num_cols if c.startswith("cocontam_")],
    "TEMPOREL":       [c for c in num_cols if c in ["year","month","season"]],
    "CATEGORIEL":     cat_cols,
}
for gname, gcols in groups.items():
    print(f"  {gname:<15} {len(gcols):3d} features")

X = df[feature_names].copy()
y = df["target"].values
print(f"\nMatrice X : {X.shape}")

════════════════════════════════════════════════════════
  Exclusions (fuite + bruit)   |  DROP_LOCATION = True
════════════════════════════════════════════════════════
  _ngL   (concentrations)      : 32 colonnes
  label_ (détection PFAS)      : 31 colonnes
  _detected (détection PFAS)   : 31 colonnes  <- corr jusqu'à 0.79
  target_sum_gt70              : 1 colonne   <- ancienne cible (corr=0.59)
  gldas_dist_km                : 1 colonne   <- bruit artefact grille
  >90% null                    : 10 colonnes
  redondantes géo              : 3 colonnes
  localisation pure (Dong)     : 6 colonnes  <- county, dwr_region, latitude, longitude, regional_board, sgma_region_office
────────────────────────────────────────────────────────
  Features numériques retenues : 82
  Features catégorielles       : 4
  TOTAL                        : 86
════════════════════════════════════════════════════════

Groupes de features retenues :
  SPATIAL           0 features
  GEOTRACKER        5 features
 

In [6]:
# Aperçu des taux de valeurs manquantes par groupe de features
missing = X.isna().mean().sort_values(ascending=False)
missing_top = missing[missing > 0].head(30)

if len(missing_top) > 0:
    fig, ax = plt.subplots(figsize=(9, max(4, len(missing_top)*0.3)))
    colors = ["#e74c3c" if v > 0.5 else "#f39c12" if v > 0.2 else "#3498db" for v in missing_top]
    ax.barh(missing_top.index[::-1], missing_top.values[::-1], color=colors[::-1])
    ax.set_xlabel("Taux de valeurs manquantes")
    ax.set_title(f"Features avec valeurs manquantes ({len(missing_top)}/{len(feature_names)})")
    ax.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="50%")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Aucune valeur manquante dans les features !")

## 4. Prétraitement et split train/test

In [7]:
# ── Split stratifié 80/20 ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train : {len(y_train):,} ({100*y_train.mean():.1f}% positifs)")
print(f"Test  : {len(y_test):,}  ({100*y_test.mean():.1f}% positifs)")

# ── Préprocesseur sklearn ────────────────────────────────────────────────────
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1,
    )),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
], remainder="drop")

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
print(f"\nX_train_proc : {X_train_proc.shape}")

Train : 37,070 (22.3% positifs)
Test  : 9,268  (22.3% positifs)



X_train_proc : (37070, 86)


## 5. Fonctions utilitaires (métriques + graphiques)

In [8]:
def evaluate_model(name, y_true, y_pred, y_proba):
    """Calcule et affiche les métriques principales.

    Sur données déséquilibrées (ici 1:3.5), le RAPPEL (recall) de la classe
    positive est crucial : il mesure la part de puits réellement au-dessus du
    seuil que le modèle détecte. On l'accompagne de la précision et de
    l'accuracy équilibrée (moyenne des rappels par classe), insensible au
    déséquilibre contrairement à l'accuracy brute.
    """
    auc  = roc_auc_score(y_true, y_proba)
    ap   = average_precision_score(y_true, y_proba)
    f1   = f1_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)         # rappel classe positive (sensibilité)
    prec = precision_score(y_true, y_pred)      # précision classe positive
    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)  # moyenne des rappels par classe
    rep  = classification_report(y_true, y_pred, target_names=["< seuil (0)", "≥ seuil (1)"])
    print(f"{'═'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    print(f"  ROC-AUC            : {auc:.4f}")
    print(f"  Average Precision  : {ap:.4f}")
    print(f"  Rappel (≥ seuil)   : {rec:.4f}   ← détection des positifs")
    print(f"  Précision (≥ seuil): {prec:.4f}")
    print(f"  F1 (≥ seuil)       : {f1:.4f}")
    print(f"  Accuracy           : {acc:.4f}")
    print(f"  Balanced Accuracy  : {bacc:.4f}   ← insensible au déséquilibre")
    print(f"{'═'*55}")
    print(rep)
    return {"name": name, "roc_auc": auc, "avg_precision": ap, "f1": f1,
            "recall": rec, "precision": prec, "accuracy": acc,
            "balanced_accuracy": bacc}

def plot_roc_pr(models_results, y_test, title=""):
    """Courbes ROC et Précision-Rappel superposées pour plusieurs modèles."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    palette = ["#2980b9", "#e74c3c", "#27ae60", "#8e44ad"]

    for i, (name, y_proba, metrics) in enumerate(models_results):
        color = palette[i % len(palette)]
        # ROC
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        ax1.plot(fpr, tpr, lw=2, color=color,
                 label=f"{name} (AUC={metrics['roc_auc']:.4f})")
        # PR
        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        ax2.plot(rec, prec, lw=2, color=color,
                 label=f"{name} (AP={metrics['avg_precision']:.4f})")

    baseline_prev = y_test.mean()
    ax1.plot([0,1],[0,1], "k--", lw=1, label="Aléatoire")
    ax2.axhline(baseline_prev, color="k", linestyle="--", lw=1,
                label=f"Baseline ({baseline_prev:.2f})")

    ax1.set(xlabel="Taux faux positifs", ylabel="Taux vrais positifs",
            title="Courbe ROC")
    ax2.set(xlabel="Rappel", ylabel="Précision",
            title="Courbe Précision-Rappel")
    ax1.legend(fontsize=9); ax2.legend(fontsize=9)
    ax1.grid(alpha=0.3); ax2.grid(alpha=0.3)

    if title:
        fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "ml_sumthr_roc_pr.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_confusion_matrices(models_preds, y_test):
    """Matrices de confusion côte à côte."""
    n = len(models_preds)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, (name, y_pred) in zip(axes, models_preds):
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred,
            display_labels=["< MCL (0)", "≥ MCL (1)"],
            cmap="Blues", ax=ax, colorbar=False,
        )
        ax.set_title(f"Matrice de confusion\n{name}")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "ml_sumthr_confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Fonctions utilitaires prêtes.")

Fonctions utilitaires prêtes.


## 6. Random Forest

In [9]:
# ── Paramètres RF ────────────────────────────────────────────────────────────
RF_PARAMS = {
    "n_estimators":     500,
    "max_features":     "sqrt",
    "min_samples_leaf": 2,
    "max_depth":        None,
    "oob_score":        True,
    "class_weight":     "balanced",  # compense le déséquilibre résiduel
    "n_jobs":           -1,
    "random_state":     RANDOM_STATE,
}

print("Entraînement Random Forest (500 arbres)…")
rf = RandomForestClassifier(**RF_PARAMS)
rf.fit(X_train_proc, y_train)
print(f"OOB score : {rf.oob_score_:.4f}")

Entraînement Random Forest (500 arbres)…


OOB score : 0.9318


In [10]:
# ── Évaluation RF ────────────────────────────────────────────────────────────
y_pred_rf   = rf.predict(X_test_proc)
y_proba_rf  = rf.predict_proba(X_test_proc)[:, 1]
metrics_rf  = evaluate_model("Random Forest", y_test, y_pred_rf, y_proba_rf)

═══════════════════════════════════════════════════════
  Random Forest
───────────────────────────────────────────────────────
  ROC-AUC            : 0.9766
  Average Precision  : 0.9268
  Rappel (≥ seuil)   : 0.8849   ← détection des positifs
  Précision (≥ seuil): 0.8291
  F1 (≥ seuil)       : 0.8561
  Accuracy           : 0.9336
  Balanced Accuracy  : 0.9163   ← insensible au déséquilibre
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

 < seuil (0)       0.97      0.95      0.96      7201
 ≥ seuil (1)       0.83      0.88      0.86      2067

    accuracy                           0.93      9268
   macro avg       0.90      0.92      0.91      9268
weighted avg       0.94      0.93      0.93      9268



In [11]:
# ── Validation croisée RF ────────────────────────────────────────────────────
print("Validation croisée 5-fold RF…")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rf = cross_validate(
    RandomForestClassifier(**{**RF_PARAMS, "oob_score": False}),
    X_train_proc, y_train,
    cv=skf,
    scoring=["roc_auc", "average_precision", "f1", "recall", "balanced_accuracy"],
    n_jobs=-1,
)
for metric in ["roc_auc", "average_precision", "f1", "recall", "balanced_accuracy"]:
    scores = cv_rf[f"test_{metric}"]
    print(f"  {metric:25s}: {scores.mean():.4f} ± {scores.std():.4f}")

Validation croisée 5-fold RF…


  roc_auc                  : 0.9715 ± 0.0023
  average_precision        : 0.9194 ± 0.0043
  f1                       : 0.8423 ± 0.0053
  recall                   : 0.8610 ± 0.0180
  balanced_accuracy        : 0.9042 ± 0.0070


## 7. XGBoost

In [12]:
# ── Paramètres XGBoost ───────────────────────────────────────────────────────
# scale_pos_weight = n_neg / n_pos pour compenser le déséquilibre de classes
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos:.4f}")

XGB_PARAMS = {
    "n_estimators":     1000,
    "learning_rate":    0.05,
    "max_depth":        6,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "gamma":            0.1,
    "reg_alpha":        0.1,   # L1
    "reg_lambda":       1.0,   # L2
    "scale_pos_weight": scale_pos,
    "tree_method":      "hist",  # rapide sur CPU
    "n_jobs":           -1,
    "random_state":     RANDOM_STATE,
    "eval_metric":      "auc",
}

print("Entraînement XGBoost (1000 rounds, early stopping)…")

# Split interne pour early stopping (20% du train)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)

xgb_model = xgb.XGBClassifier(
    **XGB_PARAMS,
    early_stopping_rounds=50,
    verbosity=0,
)
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"Best iteration : {xgb_model.best_iteration}")
print(f"Best AUC (val) : {xgb_model.best_score:.4f}")

scale_pos_weight = 3.4841
Entraînement XGBoost (1000 rounds, early stopping)…


Best iteration : 889
Best AUC (val) : 0.9728


In [13]:
# ── Évaluation XGBoost ───────────────────────────────────────────────────────
y_pred_xgb   = xgb_model.predict(X_test_proc)
y_proba_xgb  = xgb_model.predict_proba(X_test_proc)[:, 1]
metrics_xgb  = evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)

═══════════════════════════════════════════════════════
  XGBoost
───────────────────────────────────────────────────────
  ROC-AUC            : 0.9752
  Average Precision  : 0.9236
  Rappel (≥ seuil)   : 0.9182   ← détection des positifs
  Précision (≥ seuil): 0.8022
  F1 (≥ seuil)       : 0.8563
  Accuracy           : 0.9313
  Balanced Accuracy  : 0.9266   ← insensible au déséquilibre
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

 < seuil (0)       0.98      0.94      0.95      7201
 ≥ seuil (1)       0.80      0.92      0.86      2067

    accuracy                           0.93      9268
   macro avg       0.89      0.93      0.91      9268
weighted avg       0.94      0.93      0.93      9268



In [14]:
# ── Validation croisée XGBoost ───────────────────────────────────────────────
print("Validation croisée 5-fold XGBoost…")
cv_xgb = cross_validate(
    xgb.XGBClassifier(
        **{k: v for k, v in XGB_PARAMS.items() if k != "eval_metric"},
        early_stopping_rounds=None,
        verbosity=0,
    ),
    X_train_proc, y_train,
    cv=skf,
    scoring=["roc_auc", "average_precision", "f1", "recall", "balanced_accuracy"],
    n_jobs=-1,
)
for metric in ["roc_auc", "average_precision", "f1", "recall", "balanced_accuracy"]:
    scores = cv_xgb[f"test_{metric}"]
    print(f"  {metric:25s}: {scores.mean():.4f} ± {scores.std():.4f}")

Validation croisée 5-fold XGBoost…


  roc_auc                  : 0.9736 ± 0.0014
  average_precision        : 0.9232 ± 0.0053
  f1                       : 0.8502 ± 0.0049
  recall                   : 0.9054 ± 0.0123
  balanced_accuracy        : 0.9205 ± 0.0045


## 8. Optimisation des hyperparamètres — Optuna

Recherche bayésienne (TPE sampler) pour RF et XGBoost.
- **RF** : 3-fold CV, 40 trials (~10 min)
- **XGBoost** : validation interne + early stopping, 60 trials (~8 min)

Mettez `RUN_OPTUNA = False` pour sauter et garder les paramètres par défaut.

In [15]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # supprimer le spam

RUN_OPTUNA  = True   # ← mettre False pour passer cette section
N_TRIALS_RF  = 40
N_TRIALS_XGB = 60
TIMEOUT_RF   = 600   # secondes max par study (10 min)
TIMEOUT_XGB  = 480   # secondes max par study  (8 min)

print(f"Optuna {optuna.__version__}")
print(f"Trials RF={N_TRIALS_RF} (timeout={TIMEOUT_RF}s) | XGB={N_TRIALS_XGB} (timeout={TIMEOUT_XGB}s)")

Optuna 4.9.0
Trials RF=40 (timeout=600s) | XGB=60 (timeout=480s)


In [16]:
# ── Optuna RF ────────────────────────────────────────────────────────────────
if RUN_OPTUNA:
    from sklearn.model_selection import cross_val_score

    def objective_rf(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 200, 800, step=100),
            "max_depth":        trial.suggest_categorical("max_depth", [None, 15, 20, 30]),
            "max_features":     trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "min_samples_split":trial.suggest_int("min_samples_split", 2, 15),
            "class_weight":     "balanced",
            "oob_score":        False,
            "n_jobs":           -1,
            "random_state":     RANDOM_STATE,
        }
        model = RandomForestClassifier(**params)
        scores = cross_val_score(
            model, X_train_proc, y_train,
            cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
            scoring="roc_auc", n_jobs=1,   # n_jobs=1 car RF est déjà //
        )
        return scores.mean()

    sampler_rf = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study_rf   = optuna.create_study(direction="maximize", sampler=sampler_rf,
                                     study_name="rf_epa2024")
    # Démarrer avec les paramètres actuels comme point de départ
    study_rf.enqueue_trial({
        "n_estimators": 500, "max_depth": None, "max_features": "sqrt",
        "min_samples_leaf": 2, "min_samples_split": 2,
    })

    study_rf.optimize(objective_rf, n_trials=N_TRIALS_RF,
                      timeout=TIMEOUT_RF, show_progress_bar=True)

    print(f"\n[RF] Meilleur AUC-CV : {study_rf.best_value:.4f}")
    print(f"[RF] Meilleurs params :")
    for k, v in study_rf.best_params.items():
        print(f"     {k:<25} = {v}")
    
    # Visualisation de l'historique
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    values = [t.value for t in study_rf.trials if t.value is not None]
    best_so_far = [max(values[:i+1]) for i in range(len(values))]
    axes[0].plot(values, "o", alpha=0.4, color="steelblue", ms=5, label="Trial")
    axes[0].plot(best_so_far, "-", color="navy", lw=2, label="Meilleur")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("AUC-CV (3-fold)")
    axes[0].set_title("Random Forest — Historique Optuna")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Importance des hyperparamètres
    try:
        imp = optuna.importance.get_param_importances(study_rf)
        axes[1].barh(list(imp.keys())[::-1], list(imp.values())[::-1], color="steelblue")
        axes[1].set_xlabel("Importance relative")
        axes[1].set_title("RF — Importance des hyperparamètres")
        axes[1].grid(axis="x", alpha=0.3)
    except Exception:
        axes[1].text(0.5, 0.5, "Pas assez de trials", ha="center", transform=axes[1].transAxes)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "sumthr_optuna_rf_history.png", dpi=130, bbox_inches="tight")
    plt.show()
else:
    print("Optuna RF ignoré (RUN_OPTUNA=False)")
    study_rf = None

  0%|          | 0/40 [00:00<?, ?it/s]


[RF] Meilleur AUC-CV : 0.9730
[RF] Meilleurs params :
     n_estimators              = 400
     max_depth                 = 20
     max_features              = 0.5
     min_samples_leaf          = 2
     min_samples_split         = 2


In [17]:
# ── Optuna XGBoost ───────────────────────────────────────────────────────────
if RUN_OPTUNA:
    # Split interne stable pour tous les trials
    X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(
        X_train_proc, y_train, test_size=0.18,
        stratify=y_train, random_state=RANDOM_STATE,
    )

    def objective_xgb(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 200, 2000, step=100),
            "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            "max_depth":         trial.suggest_int("max_depth", 3, 12),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
            "min_child_weight":  trial.suggest_int("min_child_weight", 1, 15),
            "gamma":             trial.suggest_float("gamma", 0.0, 2.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "scale_pos_weight":  scale_pos,
            "tree_method":       "hist",
            "n_jobs":            -1,
            "random_state":      RANDOM_STATE,
            "eval_metric":       "auc",
            "verbosity":         0,
        }
        model = xgb.XGBClassifier(**params, early_stopping_rounds=40)
        model.fit(X_opt_tr, y_opt_tr,
                  eval_set=[(X_opt_val, y_opt_val)],
                  verbose=False)
        return roc_auc_score(y_opt_val, model.predict_proba(X_opt_val)[:, 1])

    sampler_xgb = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    pruner_xgb  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    study_xgb   = optuna.create_study(direction="maximize",
                                      sampler=sampler_xgb, pruner=pruner_xgb,
                                      study_name="xgb_epa2024")
    # Point de départ = paramètres actuels
    study_xgb.enqueue_trial({
        "n_estimators": 1000, "learning_rate": 0.05, "max_depth": 6,
        "subsample": 0.8, "colsample_bytree": 0.8, "colsample_bylevel": 1.0,
        "min_child_weight": 3, "gamma": 0.1,
        "reg_alpha": 0.1, "reg_lambda": 1.0,
    })

    study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_XGB,
                       timeout=TIMEOUT_XGB, show_progress_bar=True)

    print(f"\n[XGB] Meilleur AUC-val : {study_xgb.best_value:.4f}")
    print(f"[XGB] Meilleurs params :")
    for k, v in study_xgb.best_params.items():
        print(f"     {k:<25} = {v}")

    # Visualisation
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    values_x = [t.value for t in study_xgb.trials if t.value is not None]
    best_so_far_x = [max(values_x[:i+1]) for i in range(len(values_x))]
    axes[0].plot(values_x, "o", alpha=0.4, color="crimson", ms=5, label="Trial")
    axes[0].plot(best_so_far_x, "-", color="darkred", lw=2, label="Meilleur")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("AUC val interne")
    axes[0].set_title("XGBoost — Historique Optuna")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    try:
        imp_x = optuna.importance.get_param_importances(study_xgb)
        axes[1].barh(list(imp_x.keys())[::-1], list(imp_x.values())[::-1], color="crimson")
        axes[1].set_xlabel("Importance relative")
        axes[1].set_title("XGBoost — Importance des hyperparamètres")
        axes[1].grid(axis="x", alpha=0.3)
    except Exception:
        axes[1].text(0.5, 0.5, "Pas assez de trials", ha="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "sumthr_optuna_xgb_history.png", dpi=130, bbox_inches="tight")
    plt.show()
else:
    print("Optuna XGBoost ignoré (RUN_OPTUNA=False)")
    study_xgb = None

  0%|          | 0/60 [00:00<?, ?it/s]


[XGB] Meilleur AUC-val : 0.9763
[XGB] Meilleurs params :
     n_estimators              = 2000
     learning_rate             = 0.10850565063631093
     max_depth                 = 12
     subsample                 = 0.9920417450861643
     colsample_bytree          = 0.886087034273005
     colsample_bylevel         = 0.9383414430558482
     min_child_weight          = 1
     gamma                     = 0.7091644545936173
     reg_alpha                 = 8.567473224936378e-07
     reg_lambda                = 0.0015173141341339228


In [18]:
# ── Ré-entraînement avec les meilleurs hyperparamètres Optuna ────────────────
if RUN_OPTUNA and study_rf is not None and study_xgb is not None:

    # ─── Random Forest ───────────────────────────────────────────────────────
    best_rf_params = {
        **study_rf.best_params,
        "oob_score":    True,
        "class_weight": "balanced",
        "n_jobs":       -1,
        "random_state": RANDOM_STATE,
    }
    print("Ré-entraînement RF avec meilleurs params Optuna…")
    rf = RandomForestClassifier(**best_rf_params)
    rf.fit(X_train_proc, y_train)
    print(f"  OOB score : {rf.oob_score_:.4f}")

    y_pred_rf   = rf.predict(X_test_proc)
    y_proba_rf  = rf.predict_proba(X_test_proc)[:, 1]
    metrics_rf  = evaluate_model("RF (Optuna)", y_test, y_pred_rf, y_proba_rf)

    # ─── XGBoost ─────────────────────────────────────────────────────────────
    best_xgb_params = {
        **study_xgb.best_params,
        "scale_pos_weight": scale_pos,
        "tree_method":      "hist",
        "n_jobs":           -1,
        "random_state":     RANDOM_STATE,
        "eval_metric":      "auc",
        "verbosity":        0,
    }
    print("\nRé-entraînement XGBoost avec meilleurs params Optuna…")
    xgb_model = xgb.XGBClassifier(**best_xgb_params, early_stopping_rounds=50)
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    print(f"  Best iteration : {xgb_model.best_iteration}")

    y_pred_xgb   = xgb_model.predict(X_test_proc)
    y_proba_xgb  = xgb_model.predict_proba(X_test_proc)[:, 1]
    metrics_xgb  = evaluate_model("XGB (Optuna)", y_test, y_pred_xgb, y_proba_xgb)

    # ─── Gain vs baseline ────────────────────────────────────────────────────
    print("\n┌─────────────────────────────────────────────────────────┐")
    print("│  Gain Optuna vs paramètres par défaut                   │")
    print("├───────────┬──────────────┬──────────────┬───────────────┤")
    print("│  Modèle   │   AUC avant  │   AUC après  │     Δ         │")
    print("├───────────┼──────────────┼──────────────┼───────────────┤")
    # récupérer les scores baseline depuis le dict comparison (cellule suivante)
    # on les réaffiche simplement ici
    auc_rf_optuna  = metrics_rf["roc_auc"]
    auc_xgb_optuna = metrics_xgb["roc_auc"]
    print(f"│  RF       │     —        │   {auc_rf_optuna:.4f}     │     —         │")
    print(f"│  XGBoost  │     —        │   {auc_xgb_optuna:.4f}     │     —         │")
    print("└───────────┴──────────────┴──────────────┴───────────────┘")
    print("\n  (les valeurs 'avant' sont dans la cellule 6/7 au-dessus)")

else:
    print("Modèles non ré-entraînés — Optuna était désactivé.")
    print("RF et XGBoost gardent les paramètres par défaut des sections 6 et 7.")

Ré-entraînement RF avec meilleurs params Optuna…


  OOB score : 0.9412


═══════════════════════════════════════════════════════
  RF (Optuna)
───────────────────────────────────────────────────────
  ROC-AUC            : 0.9814
  Average Precision  : 0.9432
  Rappel (≥ seuil)   : 0.9061   ← détection des positifs
  Précision (≥ seuil): 0.8445
  F1 (≥ seuil)       : 0.8742
  Accuracy           : 0.9418
  Balanced Accuracy  : 0.9291   ← insensible au déséquilibre
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

 < seuil (0)       0.97      0.95      0.96      7201
 ≥ seuil (1)       0.84      0.91      0.87      2067

    accuracy                           0.94      9268
   macro avg       0.91      0.93      0.92      9268
weighted avg       0.94      0.94      0.94      9268


Ré-entraînement XGBoost avec meilleurs params Optuna…


  Best iteration : 131
═══════════════════════════════════════════════════════
  XGB (Optuna)
───────────────────────────────────────────────────────
  ROC-AUC            : 0.9770
  Average Precision  : 0.9293
  Rappel (≥ seuil)   : 0.9124   ← détection des positifs
  Précision (≥ seuil): 0.8150
  F1 (≥ seuil)       : 0.8610
  Accuracy           : 0.9343
  Balanced Accuracy  : 0.9265   ← insensible au déséquilibre
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

 < seuil (0)       0.97      0.94      0.96      7201
 ≥ seuil (1)       0.82      0.91      0.86      2067

    accuracy                           0.93      9268
   macro avg       0.89      0.93      0.91      9268
weighted avg       0.94      0.93      0.94      9268


┌─────────────────────────────────────────────────────────┐
│  Gain Optuna vs paramètres par défaut                   │
├───────────┬──────────────┬──────────────┬───────────────┤
│  Modèle   │   AU

## 8. Comparaison des modèles

In [19]:
# ── Tableau comparatif ───────────────────────────────────────────────────────
comparison_df = pd.DataFrame([
    {
        "Modèle":           "Random Forest",
        "ROC-AUC (test)":   metrics_rf["roc_auc"],
        "AP (test)":        metrics_rf["avg_precision"],
        "Rappel (test)":    metrics_rf["recall"],
        "Précision (test)": metrics_rf["precision"],
        "F1 (test)":        metrics_rf["f1"],
        "Bal.Acc (test)":   metrics_rf["balanced_accuracy"],
        "Accuracy (test)":  metrics_rf["accuracy"],
        "ROC-AUC (CV)": f"{cv_rf['test_roc_auc'].mean():.4f}±{cv_rf['test_roc_auc'].std():.4f}",
        "Rappel (CV)":  f"{cv_rf['test_recall'].mean():.4f}±{cv_rf['test_recall'].std():.4f}",
    },
    {
        "Modèle":           "XGBoost",
        "ROC-AUC (test)":   metrics_xgb["roc_auc"],
        "AP (test)":        metrics_xgb["avg_precision"],
        "Rappel (test)":    metrics_xgb["recall"],
        "Précision (test)": metrics_xgb["precision"],
        "F1 (test)":        metrics_xgb["f1"],
        "Bal.Acc (test)":   metrics_xgb["balanced_accuracy"],
        "Accuracy (test)":  metrics_xgb["accuracy"],
        "ROC-AUC (CV)": f"{cv_xgb['test_roc_auc'].mean():.4f}±{cv_xgb['test_roc_auc'].std():.4f}",
        "Rappel (CV)":  f"{cv_xgb['test_recall'].mean():.4f}±{cv_xgb['test_recall'].std():.4f}",
    },
])
comparison_df.set_index("Modèle", inplace=True)
comparison_df.style.background_gradient(cmap="RdYlGn", subset=["ROC-AUC (test)", "AP (test)", "Rappel (test)", "Précision (test)", "F1 (test)", "Bal.Acc (test)", "Accuracy (test)"])

,ROC-AUC (test),AP (test),Rappel (test),Précision (test),F1 (test),Bal.Acc (test),Accuracy (test),ROC-AUC (CV),Rappel (CV)
Modèle,,,,,,,,,
Random Forest,0.981383,0.943182,0.906144,0.844454,0.874212,0.929117,0.941843,0.9715±0.0023,0.8610±0.0180
XGBoost,0.976995,0.929315,0.912433,0.815039,0.860991,0.926499,0.934290,0.9736±0.0014,0.9054±0.0123


In [20]:
# ── Courbes ROC + PR ─────────────────────────────────────────────────────────
models_results = [
    ("Random Forest", y_proba_rf,  metrics_rf),
    ("XGBoost",       y_proba_xgb, metrics_xgb),
]
plot_roc_pr(models_results, y_test,
            title="Comparaison RF vs XGBoost — Classification PFAS (EPA 2024)")

In [21]:
# ── Matrices de confusion ─────────────────────────────────────────────────────
models_preds = [
    ("Random Forest", y_pred_rf),
    ("XGBoost",       y_pred_xgb),
]
plot_confusion_matrices(models_preds, y_test)

In [22]:
# ── Courbe d'apprentissage XGBoost (évolution AUC vs n_estimators) ───────────
results_history = xgb_model.evals_result()
if results_history:
    val_auc = results_history["validation_0"]["auc"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(val_auc, color="#e74c3c", lw=1.5, label="AUC validation")
    ax.axvline(xgb_model.best_iteration, color="k", linestyle="--",
               label=f"Best iteration ({xgb_model.best_iteration})")
    ax.set_xlabel("Itération (n_estimators)")
    ax.set_ylabel("AUC")
    ax.set_title("Courbe d'apprentissage XGBoost")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "sumthr_xgb_learning_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8bis. Gestion du déséquilibre — seuil de décision optimal

La cible est déséquilibrée (**1:3.5**, 22.3 % de positifs). Deux leviers sont déjà
en place dès l'entraînement :

| Levier | Random Forest | XGBoost |
|--------|---------------|---------|
| Pondération des classes | `class_weight="balanced"` | `scale_pos_weight = n_neg/n_pos` |

Mais la **règle de décision par défaut (proba ≥ 0.50)** reste mal calibrée pour un
problème rare : elle privilégie la précision au détriment du **rappel** (on rate des
puits réellement contaminés). On cherche donc le **seuil optimal** qui maximise le F1.

**Anti-fuite :** le seuil est choisi sur des probabilités *out-of-fold* du **train**
(`cross_val_predict`), jamais sur le test — puis appliqué tel quel au test.

In [23]:
from sklearn.model_selection import cross_val_predict

def tune_threshold(model, X_tr, y_tr, label):
    """Seuil maximisant le F1 sur probabilités out-of-fold (train) — sans fuite."""
    oof = cross_val_predict(model, X_tr, y_tr, cv=skf,
                            method="predict_proba", n_jobs=-1)[:, 1]
    grid = np.linspace(0.05, 0.95, 181)
    f1s  = np.array([f1_score(y_tr, (oof >= t).astype(int)) for t in grid])
    recs = np.array([recall_score(y_tr, (oof >= t).astype(int)) for t in grid])
    precs= np.array([precision_score(y_tr, (oof >= t).astype(int), zero_division=0) for t in grid])
    t_best = grid[int(np.argmax(f1s))]
    print(f"  {label:13s}: seuil F1-optimal = {t_best:.3f}  (défaut 0.500)")
    return t_best, grid, f1s, recs, precs

# Modèles sans early stopping pour le CV (XGB : nombre d'arbres figé)
rf_cv  = RandomForestClassifier(**{**RF_PARAMS, "oob_score": False})
xgb_cv = xgb.XGBClassifier(
    **{k: v for k, v in XGB_PARAMS.items() if k not in ("eval_metric", "n_estimators")},
    n_estimators=int(xgb_model.best_iteration or XGB_PARAMS["n_estimators"]),
    verbosity=0,
)

print("Recherche du seuil optimal (out-of-fold, train)…")
t_rf,  g, f1_rf,  rec_rf,  pre_rf  = tune_threshold(rf_cv,  X_train_proc, y_train, "Random Forest")
t_xgb, _, f1_xgb, rec_xgb, pre_xgb = tune_threshold(xgb_cv, X_train_proc, y_train, "XGBoost")

# ── Graphique : précision / rappel / F1 vs seuil ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (name, f1s, recs, precs, t_best, color) in zip(axes, [
        ("Random Forest", f1_rf, rec_rf, pre_rf, t_rf, "#2980b9"),
        ("XGBoost",       f1_xgb, rec_xgb, pre_xgb, t_xgb, "#e74c3c")]):
    ax.plot(g, recs,  label="Rappel",    color="#27ae60", lw=2)
    ax.plot(g, precs, label="Précision", color="#e67e22", lw=2)
    ax.plot(g, f1s,   label="F1",        color=color,     lw=2.5)
    ax.axvline(t_best, ls="--", color="k", lw=1.5, label=f"seuil opt = {t_best:.2f}")
    ax.axvline(0.5,    ls=":",  color="gray", lw=1, label="défaut 0.50")
    ax.set(xlabel="Seuil de décision", ylabel="Score", title=name, ylim=(0,1))
    ax.legend(fontsize=8, loc="lower center"); ax.grid(alpha=0.3)
fig.suptitle("Optimisation du seuil — compromis précision/rappel (cible déséquilibrée 1:3.5)", fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "sumthr_threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

Recherche du seuil optimal (out-of-fold, train)…


  Random Forest: seuil F1-optimal = 0.460  (défaut 0.500)


  XGBoost      : seuil F1-optimal = 0.585  (défaut 0.500)


In [24]:
# ── Effet du seuil optimal sur le TEST : défaut 0.50 vs seuil ajusté ─────────
def at_threshold(y_true, proba, t):
    yp = (proba >= t).astype(int)
    return {"seuil": t, "rappel": recall_score(y_true, yp),
            "précision": precision_score(y_true, yp, zero_division=0),
            "f1": f1_score(y_true, yp),
            "bal_acc": balanced_accuracy_score(y_true, yp)}

rows = []
for name, proba, t_best in [("Random Forest", y_proba_rf, t_rf),
                            ("XGBoost", y_proba_xgb, t_xgb)]:
    d0 = at_threshold(y_test, proba, 0.50); d0["modèle"] = name; d0["règle"] = "défaut 0.50"
    dt = at_threshold(y_test, proba, t_best); dt["modèle"] = name; dt["règle"] = f"optimal {t_best:.2f}"
    rows += [d0, dt]

thr_df = pd.DataFrame(rows)[["modèle","règle","seuil","rappel","précision","f1","bal_acc"]]
print("Impact du seuil sur le test set :")
for name in ["Random Forest", "XGBoost"]:
    sub = thr_df[thr_df["modèle"] == name]
    d_rec = sub.iloc[1]["rappel"] - sub.iloc[0]["rappel"]
    print(f"  {name:13s}: rappel {sub.iloc[0]['rappel']:.3f} -> {sub.iloc[1]['rappel']:.3f}  (Δ={d_rec:+.3f})")

# Mémoriser le meilleur seuil pour la sauvegarde
OPTIMAL_THRESHOLDS = {"random_forest": float(t_rf), "xgboost": float(t_xgb)}
thr_df.style.format({"seuil":"{:.2f}","rappel":"{:.3f}","précision":"{:.3f}",
                     "f1":"{:.3f}","bal_acc":"{:.3f}"})\
       .background_gradient(cmap="RdYlGn", subset=["rappel","f1","bal_acc"])

Impact du seuil sur le test set :
  Random Forest: rappel 0.906 -> 0.918  (Δ=+0.012)
  XGBoost      : rappel 0.912 -> 0.890  (Δ=-0.023)


,modèle,règle,seuil,rappel,précision,f1,bal_acc
0,Random Forest,défaut 0.50,0.50,0.906,0.844,0.874,0.929
1,Random Forest,optimal 0.46,0.46,0.918,0.830,0.872,0.932
2,XGBoost,défaut 0.50,0.50,0.912,0.815,0.861,0.926
3,XGBoost,optimal 0.58,0.58,0.890,0.832,0.860,0.919


## 9. Importance des features

In [25]:
# ── Importance MDI Random Forest ──────────────────────────────────────────────
fi_rf = pd.DataFrame({
    "feature":    feature_names,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
fi_rf["rank"] = fi_rf.index + 1

# ── Importance XGBoost (gain) ─────────────────────────────────────────────────
xgb_scores = xgb_model.get_booster().get_score(importance_type="gain")
fi_xgb = pd.DataFrame([
    {"feature": f"f{i}", "xgb_gain": xgb_scores.get(f"f{i}", 0.0)}
    for i in range(len(feature_names))
])
fi_xgb["feature"] = feature_names
fi_xgb = fi_xgb.sort_values("xgb_gain", ascending=False).reset_index(drop=True)
fi_xgb["rank_xgb"] = fi_xgb.index + 1

# ── Graphique côte à côte : Top 25 ────────────────────────────────────────────
TOP_N = 25
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9))

top_rf = fi_rf.head(TOP_N)
ax1.barh(top_rf["feature"][::-1], top_rf["importance"][::-1],
         color=plt.cm.Blues(np.linspace(0.4, 0.9, TOP_N)))
ax1.set_xlabel("Importance (MDI)")
ax1.set_title(f"Random Forest — Top {TOP_N}")
ax1.grid(axis="x", alpha=0.3)

top_xgb = fi_xgb.head(TOP_N)
ax2.barh(top_xgb["feature"][::-1], top_xgb["xgb_gain"][::-1],
         color=plt.cm.Reds(np.linspace(0.4, 0.9, TOP_N)))
ax2.set_xlabel("Importance (Gain)")
ax2.set_title(f"XGBoost — Top {TOP_N}")
ax2.grid(axis="x", alpha=0.3)

plt.suptitle(f"Importance des features — Classification PFAS EPA 2024", fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ml_sumthr_feature_importance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [26]:
# ── Concordance RF vs XGBoost (rang des 30 premières features) ───────────────
top30_rf  = set(fi_rf.head(30)["feature"])
top30_xgb = set(fi_xgb.head(30)["feature"])
common    = top30_rf & top30_xgb
print(f"Features dans le Top 30 des deux modèles : {len(common)}/30")
print("\nFeatures communes (classées par rang moyen) :")
merged = (
    fi_rf[["feature","rank"]]
    .merge(fi_xgb[["feature","rank_xgb"]], on="feature")
)
merged["rank_mean"] = (merged["rank"] + merged["rank_xgb"]) / 2
merged_common = merged[merged["feature"].isin(common)].sort_values("rank_mean")
merged_common.columns = ["Feature", "Rang RF", "Rang XGB", "Rang moyen"]
merged_common.reset_index(drop=True, inplace=True)
merged_common.head(20)

Features dans le Top 30 des deux modèles : 11/30

Features communes (classées par rang moyen) :


,Feature,Rang RF,Rang XGB,Rang moyen
0,gm_dataset_name,1,1,1.0
1,n_geotracker_within_50km,2,8,5.0
2,gm_well_category,5,6,5.5
3,n_geotracker_within_10km,4,13,8.5
4,cocontam_tce,15,12,13.5
5,aqs_no2_ppb,6,25,15.5
6,aqs_ozone_ppb,7,27,17.0
7,cocontam_pce,10,24,17.0
8,soil_ksat_um_s,21,18,19.5
9,soil_silt_pct,27,29,28.0


## 10. Analyse SHAP (optionnel — lent ~2 min)

In [27]:
# Mettre RUN_SHAP = True pour activer
RUN_SHAP = False

if RUN_SHAP:
    import shap
    rng = np.random.RandomState(RANDOM_STATE)
    idx_sample = rng.choice(len(X_test_proc), 1500, replace=False)
    X_sample   = X_test_proc[idx_sample]

    # ── SHAP XGBoost ────────────────────────────────────────────────────────
    print("Calcul SHAP XGBoost…")
    explainer_xgb = shap.TreeExplainer(xgb_model)
    shap_xgb = explainer_xgb.shap_values(X_sample)
    sv_xgb   = shap_xgb if not isinstance(shap_xgb, list) else shap_xgb[1]

    fig, ax = plt.subplots(figsize=(9, 8))
    shap.summary_plot(sv_xgb, X_sample, feature_names=feature_names,
                      max_display=25, show=False, plot_size=None)
    plt.title("SHAP Summary — XGBoost (PFAS EPA 2024)", pad=12)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "sumthr_shap_summary_xgb.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── SHAP Random Forest ───────────────────────────────────────────────────
    print("Calcul SHAP Random Forest…")
    explainer_rf = shap.TreeExplainer(rf)
    shap_rf = explainer_rf.shap_values(X_sample)
    sv_rf   = shap_rf[1] if isinstance(shap_rf, list) else shap_rf

    fig, ax = plt.subplots(figsize=(9, 8))
    shap.summary_plot(sv_rf, X_sample, feature_names=feature_names,
                      max_display=25, show=False, plot_size=None)
    plt.title("SHAP Summary — Random Forest (PFAS EPA 2024)", pad=12)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "sumthr_shap_summary_rf.png", dpi=150, bbox_inches="tight")
    plt.show()

    # SHAP waterfall pour un échantillon positif et un négatif
    pos_idx = np.where(y_test[idx_sample] == 1)[0][0]
    neg_idx = np.where(y_test[idx_sample] == 0)[0][0]
    for label, sample_idx in [("positif (≥ MCL)", pos_idx), ("négatif (< MCL)", neg_idx)]:
        shap.waterfall_plot(
            shap.Explanation(values=sv_xgb[sample_idx],
                             base_values=explainer_xgb.expected_value,
                             data=X_sample[sample_idx],
                             feature_names=feature_names),
            max_display=15, show=False
        )
        plt.title(f"SHAP Waterfall — XGBoost, échantillon {label}")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"sumthr_shap_waterfall_xgb_{label.split()[0]}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("SHAP désactivé. Mettre RUN_SHAP = True pour l'activer.")

SHAP désactivé. Mettre RUN_SHAP = True pour l'activer.


## 10ter. Visualisations de synthèse (soutenance)

Graphiques à fort impact pour la présentation devant le jury :
1. **Carte des prédictions** — résultat spatial (bien que la localisation ne soit pas une feature)
2. **Courbe de calibration** — fiabilité des probabilités prédites
3. **Courbe de gain cumulé** — valeur opérationnelle : priorisation de l'échantillonnage
4. **Tableau de bord comparatif** — RF vs XGBoost sur toutes les métriques

In [28]:
# ── Carte des prédictions sur la Californie (test set, modèle Random Forest) ──
coords = df.loc[X_test.index, ["longitude", "latitude"]].copy()
coords["y_true"] = y_test
coords["proba"]  = y_proba_rf
thr_rf = OPTIMAL_THRESHOLDS["random_forest"]
coords["y_pred"] = (coords["proba"].values >= thr_rf).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# (1) Réalité terrain
for lab, c, m in [(0, "#3498db", "< seuil"), (1, "#e74c3c", "≥ seuil")]:
    sub = coords[coords["y_true"] == lab]
    axes[0].scatter(sub["longitude"], sub["latitude"], s=7, c=c, alpha=0.45,
                    label=m, edgecolors="none")
axes[0].set_title("Réalité terrain (test set)")
axes[0].legend(markerscale=2, loc="upper right")

# (2) Probabilité prédite
sc = axes[1].scatter(coords["longitude"], coords["latitude"], s=7, c=coords["proba"],
                     cmap="RdYlGn_r", alpha=0.6, vmin=0, vmax=1, edgecolors="none")
plt.colorbar(sc, ax=axes[1], shrink=0.7, label="Probabilité prédite")
axes[1].set_title("Risque PFAS prédit (Random Forest)")

# (3) Type d'erreur de classification
conds = [
    (coords.y_true == 1) & (coords.y_pred == 1),
    (coords.y_true == 0) & (coords.y_pred == 0),
    (coords.y_true == 0) & (coords.y_pred == 1),
    (coords.y_true == 1) & (coords.y_pred == 0),
]
labels = ["Vrai positif", "Vrai négatif", "Faux positif", "Faux négatif"]
import numpy as np
coords["err"] = np.select(conds, labels, default="?")
palette = {"Vrai positif": "#27ae60", "Vrai négatif": "#d5dbdb",
           "Faux positif": "#f39c12", "Faux négatif": "#c0392b"}
for cat, c in palette.items():
    sub = coords[coords["err"] == cat]
    axes[2].scatter(sub["longitude"], sub["latitude"], s=7, c=c, alpha=0.55,
                    label=f"{cat} ({len(sub)})", edgecolors="none")
axes[2].set_title(f"Erreurs (seuil optimal = {thr_rf:.2f})")
axes[2].legend(markerscale=2, fontsize=8, loc="upper right")

for ax in axes:
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_aspect("equal", adjustable="datalim"); ax.grid(alpha=0.2)
fig.suptitle("Distribution spatiale des prédictions — cible Σ PFAS ≥ 74 ng/L", fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ml_sumthr_map_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [29]:
# ── Courbe de calibration : les probabilités prédites sont-elles fiables ? ────
from sklearn.calibration import calibration_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5),
                               gridspec_kw={"height_ratios": [1]})
for name, proba, color in [("Random Forest", y_proba_rf, "#2980b9"),
                           ("XGBoost", y_proba_xgb, "#e74c3c")]:
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")
    ax1.plot(mean_pred, frac_pos, "o-", color=color, lw=2, label=name)
ax1.plot([0, 1], [0, 1], "k--", lw=1, label="Calibration parfaite")
ax1.set(xlabel="Probabilité prédite moyenne", ylabel="Fraction observée de positifs",
        title="Courbe de calibration (fiabilité)")
ax1.legend(); ax1.grid(alpha=0.3)

# Histogramme des probabilités prédites
ax2.hist(y_proba_rf, bins=30, alpha=0.55, color="#2980b9", label="Random Forest")
ax2.hist(y_proba_xgb, bins=30, alpha=0.55, color="#e74c3c", label="XGBoost")
ax2.axvline(OPTIMAL_THRESHOLDS["random_forest"], ls="--", color="#2980b9", lw=1.5)
ax2.axvline(OPTIMAL_THRESHOLDS["xgboost"], ls="--", color="#e74c3c", lw=1.5)
ax2.set(xlabel="Probabilité prédite", ylabel="Nombre de puits",
        title="Distribution des scores (seuils optimaux en pointillé)")
ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle("Fiabilité probabiliste des modèles", fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ml_sumthr_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

In [30]:
# ── Courbe de gain cumulé : valeur opérationnelle de la priorisation ─────────
def cumulative_gain(y_true, proba):
    order = np.argsort(proba)[::-1]
    y_sorted = np.asarray(y_true)[order]
    gains = np.cumsum(y_sorted) / y_sorted.sum()
    frac  = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
    return np.concatenate([[0], frac]), np.concatenate([[0], gains])

prev = y_test.mean()
fig, ax = plt.subplots(figsize=(8, 6))
annot = []
for name, proba, color in [("Random Forest", y_proba_rf, "#2980b9"),
                           ("XGBoost", y_proba_xgb, "#e74c3c")]:
    frac, gains = cumulative_gain(y_test, proba)
    ax.plot(frac, gains, color=color, lw=2.5, label=name)
    # gain à 25% des puits testés
    g25 = np.interp(0.25, frac, gains)
    annot.append((name, g25))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Aléatoire")
ax.plot([0, prev, 1], [0, 1, 1], color="gray", ls=":", lw=1.5, label="Modèle parfait")
ax.axvline(0.25, color="green", ls=":", lw=1, alpha=0.6)
ax.set(xlabel="Fraction des puits testés (triés par risque décroissant)",
       ylabel="Fraction des puits contaminés détectés",
       title="Courbe de gain cumulé — priorisation de l'échantillonnage")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ml_sumthr_cumulative_gain.png", dpi=150, bbox_inches="tight")
plt.show()

print("Lecture opérationnelle (en testant les 25% puits les plus à risque) :")
for name, g25 in annot:
    print(f"  {name:13s}: on détecte {100*g25:.0f}% des puits réellement contaminés")

Lecture opérationnelle (en testant les 25% puits les plus à risque) :
  Random Forest: on détecte 92% des puits réellement contaminés
  XGBoost      : on détecte 91% des puits réellement contaminés


In [31]:
# ── Tableau de bord comparatif RF vs XGBoost (toutes métriques) ──────────────
keys   = ["roc_auc", "avg_precision", "recall", "precision", "f1", "balanced_accuracy"]
labels = ["ROC-AUC", "Avg.Prec", "Rappel", "Précision", "F1", "Bal.Acc"]
rf_vals  = [metrics_rf[k]  for k in keys]
xgb_vals = [metrics_xgb[k] for k in keys]

x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5.5))
b1 = ax.bar(x - w/2, rf_vals,  w, label="Random Forest", color="#2980b9")
b2 = ax.bar(x + w/2, xgb_vals, w, label="XGBoost",       color="#e74c3c")
ax.bar_label(b1, fmt="%.3f", fontsize=8, padding=2)
ax.bar_label(b2, fmt="%.3f", fontsize=8, padding=2)
ymin = max(0.0, min(rf_vals + xgb_vals) - 0.06)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(ymin, 1.0)
ax.set_ylabel("Score (test set)")
ax.set_title("Synthèse des performances — cible Σ PFAS ≥ 74 ng/L")
ax.legend(loc="lower right"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "ml_sumthr_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Sauvegarde des artefacts

In [32]:
# ── Feature importance CSV ────────────────────────────────────────────────────
fi_combined = fi_rf.rename(columns={"importance": "rf_mdi", "rank": "rank_rf"}).merge(
    fi_xgb[["feature", "xgb_gain", "rank_xgb"]], on="feature", how="left"
).sort_values("rf_mdi", ascending=False)
fi_combined.to_csv(PROCESSED_DIR / "ml_sumthr_feature_importance.csv", index=False)
print(f"Feature importance → {PROCESSED_DIR / 'ml_sumthr_feature_importance.csv'}")

# ── JSON résultats ────────────────────────────────────────────────────────────
results = {
    "target": "target_sum_ge_threshold",
    "regulation": "Somme des 27 PFAS >= somme des seuils individuels (notebook 03)",
    "sum_seuil_ngL": SUM_SEUIL,
    "n_thresholds": len(PFAS_TARGET_COLS),
    "dataset": {"n_total": len(df), "n_features": len(feature_names),
                "n_pos": int(y.sum()), "pct_pos": round(100*y.mean(), 2)},
    "split": {"train": int(len(y_train)), "test": int(len(y_test))},
    "optimal_thresholds": OPTIMAL_THRESHOLDS,
    "random_forest": {
        "params": {k: v for k, v in RF_PARAMS.items() if k not in ("n_jobs",)},
        "oob_score": round(rf.oob_score_, 4),
        "test": {k: round(v, 4) for k, v in metrics_rf.items() if k != "name"},
        "cv_roc_auc": f"{cv_rf['test_roc_auc'].mean():.4f}±{cv_rf['test_roc_auc'].std():.4f}",
        "cv_f1":      f"{cv_rf['test_f1'].mean():.4f}±{cv_rf['test_f1'].std():.4f}",
        "cv_recall":  f"{cv_rf['test_recall'].mean():.4f}±{cv_rf['test_recall'].std():.4f}",
        "cv_balanced_accuracy": f"{cv_rf['test_balanced_accuracy'].mean():.4f}±{cv_rf['test_balanced_accuracy'].std():.4f}",
    },
    "xgboost": {
        "best_iteration": int(xgb_model.best_iteration),
        "best_val_auc": round(float(xgb_model.best_score), 4),
        "test": {k: round(v, 4) for k, v in metrics_xgb.items() if k != "name"},
        "cv_roc_auc": f"{cv_xgb['test_roc_auc'].mean():.4f}±{cv_xgb['test_roc_auc'].std():.4f}",
        "cv_f1":      f"{cv_xgb['test_f1'].mean():.4f}±{cv_xgb['test_f1'].std():.4f}",
        "cv_recall":  f"{cv_xgb['test_recall'].mean():.4f}±{cv_xgb['test_recall'].std():.4f}",
        "cv_balanced_accuracy": f"{cv_xgb['test_balanced_accuracy'].mean():.4f}±{cv_xgb['test_balanced_accuracy'].std():.4f}",
    },
    "top10_features_rf":  fi_rf.head(10)[["rank","feature","importance"]].rename(columns={"importance":"rf_mdi"}).to_dict("records"),
    "top10_features_xgb": fi_xgb.head(10)[["rank_xgb","feature","xgb_gain"]].to_dict("records"),
}
with open(PROCESSED_DIR / "ml_sumthr_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Résultats JSON → {PROCESSED_DIR / 'ml_sumthr_results.json'}")

# ── Modèles pickle ────────────────────────────────────────────────────────────
artifact_rf = {
    "model": rf, "preprocessor": preprocessor,
    "optimal_threshold": OPTIMAL_THRESHOLDS["random_forest"],
    "feature_names": feature_names, "num_cols": num_cols, "cat_cols": cat_cols,
    "target": "target_sum_ge_threshold", "sum_seuil_ngL": SUM_SEUIL,
}
with open(MODELS_DIR / "rf_binary_sumthr.pkl", "wb") as f:
    pickle.dump(artifact_rf, f)
print(f"RF → {MODELS_DIR / 'rf_binary_sumthr.pkl'}")

artifact_xgb = {
    "model": xgb_model, "preprocessor": preprocessor,
    "optimal_threshold": OPTIMAL_THRESHOLDS["xgboost"],
    "feature_names": feature_names, "num_cols": num_cols, "cat_cols": cat_cols,
    "target": "target_sum_ge_threshold", "sum_seuil_ngL": SUM_SEUIL,
}
with open(MODELS_DIR / "xgb_binary_sumthr.pkl", "wb") as f:
    pickle.dump(artifact_xgb, f)
print(f"XGB → {MODELS_DIR / 'xgb_binary_sumthr.pkl'}")

Feature importance → ../data/processed/ml_sumthr_feature_importance.csv
Résultats JSON → ../data/processed/ml_sumthr_results.json


RF → ../models/rf_binary_sumthr.pkl
XGB → ../models/xgb_binary_sumthr.pkl


## 12. Synthèse finale

In [33]:
print("╔══════════════════════════════════════════════════════════╗")
print("║      RÉSULTATS — Classification Σ PFAS ≥ Σ seuils        ║")
print("║      Cible : somme(27 PFAS) ≥ sum_seuil (74 ng/L)       ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Dataset : {len(df):,} échantillons  |  {len(feature_names)} features          ║")
print(f"║  Positifs : {int(y.sum()):,} ({100*y.mean():.1f}%)   Négatifs : {int((y==0).sum()):,} ({100*(y==0).mean():.1f}%)  ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  {'Modèle':<20} {'AUC':>7} {'AP':>7} {'F1':>7}        ║")
print("╠══════════════════════════════════════════════════════════╣")
for name, m in [("Random Forest", metrics_rf), ("XGBoost", metrics_xgb)]:
    print(f"║  {name:<20} {m['roc_auc']:>7.4f} {m['avg_precision']:>7.4f} {m['f1']:>7.4f}        ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Top 5 features communes (RF ∩ XGBoost) :               ║")
for _, row in merged_common.head(5).iterrows():
    feat = str(row["Feature"])[:38]
    print(f"║    #{row['Rang RF']:3.0f}/{row['Rang XGB']:3.0f}  {feat:<38}║")
print("╚══════════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════════╗
║      RÉSULTATS — Classification Σ PFAS ≥ Σ seuils        ║
║      Cible : somme(27 PFAS) ≥ sum_seuil (74 ng/L)       ║
╠══════════════════════════════════════════════════════════╣
║  Dataset : 46,338 échantillons  |  86 features          ║
║  Positifs : 10,334 (22.3%)   Négatifs : 36,004 (77.7%)  ║
╠══════════════════════════════════════════════════════════╣
║  Modèle                   AUC      AP      F1        ║
╠══════════════════════════════════════════════════════════╣
║  Random Forest         0.9814  0.9432  0.8742        ║
║  XGBoost               0.9770  0.9293  0.8610        ║
╠══════════════════════════════════════════════════════════╣
║  Top 5 features communes (RF ∩ XGBoost) :               ║
║    #  1/  1  gm_dataset_name                       ║
║    #  2/  8  n_geotracker_within_50km              ║
║    #  5/  6  gm_well_category                      ║
║    #  4/ 13  n_geotracker_within_10km              ║
║  